# p53–DNA (`2AC0_dimer`) & MDM2–RPL11 (`4XXB`) OpenMM 8 MD Runner

Reproducible **AmberTools (`ff14SB` + `DNA.OL15` + `OPC` + bonded `ZAFF` $\mathrm{Zn^{2+}}$ + ParmEd 4 fs HMR) + OpenMM 8 (CUDA)** runner with cross-GPU portable `state.xml` checkpointing on Google Drive.

**Usage notes:**
1. Sync this repository to `DRIVE_PIPELINE_DIR` in Google Drive before launching.
2. If `build/solv_hmr.prmtop` is already present in `runs/<SYSTEM>_<VARIANT>/build/`, topology building is skipped automatically so parallel replica tabs share identical topologies.
3. If a Colab runtime disconnects, reconnect (to any CUDA GPU) and click **Run all**: `02_run_md.py` automatically resumes from the latest `5 ns` `state.xml` checkpoint.
4. After `prod_dry.dcd` is written upon completion, remove the full solvated `prod.dcd` (`~10–13 GB`) from Drive to reclaim storage.

In [ ]:
#@title 1. Configure Experiment & Replica Parameters
SYSTEM = "4XXB" #@param ["4XXB", "2AC0_dimer"]
MUTATIONS = "WT" #@param {type:"string"}
# Examples:
#   4XXB MDM2 mutant     : "B:R326H" or "B:R326C"
#   2AC0 homozygous mut  : "A:R280K,B:R280K" or "A:R280T,B:R280T"
#   2AC0 heterozygous    : "A:R280K"
VARIANT_NAME = "WT" #@param {type:"string"}
REPLICA_ID = 1 #@param [1, 2, 3] {type:"raw"}
TARGET_NS = 200.0 #@param {type:"number"}
CHUNK_NS = 5.0 #@param {type:"number"}
SAVE_PS = 20.0 #@param {type:"number"}
DT_FS = 4.0 #@param [4.0, 2.0] {type:"raw"}
DRIVE_PIPELINE_DIR = "/content/drive/MyDrive/md-p53-mdm2" #@param {type:"string"}

print(f"Configured: {SYSTEM} | variant={VARIANT_NAME} ({MUTATIONS}) | rep={REPLICA_ID} | target={TARGET_NS} ns (dt={DT_FS} fs)")


In [ ]:
#@title 2. Mount Google Drive & Verify GPU
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

pipe_dir = Path(DRIVE_PIPELINE_DIR)
if not pipe_dir.exists():
    raise FileNotFoundError(f"Pipeline directory not found at {pipe_dir}. Update DRIVE_PIPELINE_DIR above.")

RUN_DIR = pipe_dir / "runs" / f"{SYSTEM}_{VARIANT_NAME}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Pipeline dir : {pipe_dir}")
print(f"[OK] Run dir      : {RUN_DIR}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
#@title 3. Install OpenMM 8 + AmberTools via Micromamba (~2.5 min)
import subprocess

setup_sh = """
set -e
if [ ! -x /usr/local/envs/md/bin/python ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local/bin --strip-components=1 bin/micromamba
  micromamba create -y -p /usr/local/envs/md -c conda-forge \
    python=3.11 "openmm>=8.1" cuda-version=12.4 pdbfixer ambertools parmed mdtraj pymbar matplotlib pandas scipy
else
  echo "[CACHED] /usr/local/envs/md already installed."
fi
/usr/local/envs/md/bin/python -c "import openmm; print('OpenMM:', openmm.__version__, 'Platforms:', [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())])"
"""
subprocess.run(["bash", "-lc", setup_sh], check=True)


In [ ]:
#@title 4. Build Topology (Skipped Automatically if Already Built)
import subprocess
hmr_top = RUN_DIR / "build" / "solv_hmr.prmtop"
if hmr_top.exists():
    print(f"[SKIP] Found existing topology: {hmr_top}")
else:
    cmd = [
        "/usr/local/envs/md/bin/python",
        str(pipe_dir / "scripts" / "01_build_system.py"),
        "--system", SYSTEM,
        "--mutations", MUTATIONS,
        "--variant", VARIANT_NAME,
        "--outdir", str(RUN_DIR),
        "--params-dir", str(pipe_dir / "params"),
    ]
    print("[RUN]", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
#@title 5. Run 10-Step Relaxation & Production MD (Auto-Resumes via state.xml)
import subprocess
cmd = [
    "/usr/local/envs/md/bin/python",
    str(pipe_dir / "scripts" / "02_run_md.py"),
    "--run-dir", str(RUN_DIR),
    "--replica", str(REPLICA_ID),
    "--ns", str(TARGET_NS),
    "--chunk-ns", str(CHUNK_NS),
    "--save-ps", str(SAVE_PS),
    "--dt-fs", str(DT_FS),
    "--platform", "CUDA",
]
print("[RUN]", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 6. Monitor All Replicas & Optional Analysis / MM-GBSA
import json
for r in sorted((pipe_dir / "runs").glob("*/rep*/status.json")):
    d = json.loads(r.read_text())
    print(f"{r.parent.parent.name}/{r.parent.name}: {d['completed_ns']}/{d['target_ns']} ns (~{d.get('est_ns_per_day', 0)} ns/day) | {d.get('updated_utc')}")

# Uncomment once both WT and Mutant replicas complete:
# !/usr/local/envs/md/bin/python "{pipe_dir}/scripts/03_analyze_and_mmpbsa.py" \
#     --system {SYSTEM} \
#     --wt-dir "{pipe_dir}/runs/{SYSTEM}_WT" \
#     --mut-dir "{pipe_dir}/runs/{SYSTEM}_{VARIANT_NAME}" \
#     --outdir "{pipe_dir}/analysis/{SYSTEM}_{VARIANT_NAME}" \
#     --run-mmpbsa
